# Lexer e parser da linguagem LARP

Analisador lexico (Flex) e sintatico (Bison) para uma linguagem em estilo C com
declaracoes, atribuicoes, `if`/`else`, ternario (`if (...) -> ... else -> ...`) e
`route`/`fall`.

- O **lexer** (`clarker.l`) reconhece os tokens, ignora os espacos em branco e reporta
  com o numero da linha os erros lexicos.
- O **parser** (`parser.y`) recebe os tokens do lexer, verifica a estrutura do programa e
  faz a analise semantica. Todo erro e reportado com a linha. Se o programa for valido,
  imprime a tabela de simbolos, a traducao de cada comando para notacao pos-fixada e
  `Programa sintatica e semanticamente correto`.

O lexer tambem pode rodar sozinho: nesse caso imprime a sequencia de tokens, a tabela de
simbolos e um resumo.

## 1. Clonar o repositorio

In [10]:
import os

REPO = 'https://github.com/Rafaeruo/parser-larp.git'
DIR = '/content/parser-larp'

if not os.path.exists(DIR):
    !git clone $REPO $DIR

%cd $DIR
!git pull --quiet && echo 'repositorio atualizado'

!apt-get -qq install -y tree
!tree

/content/parser-larp
repositorio atualizado
.
├── bin
│   ├── lexer
│   ├── lex.yy.c
│   ├── parser
│   ├── parser.tab.c
│   └── parser.tab.h
├── clarker.l
├── colab.ipynb
├── gramatica_ebnf.txt
├── Makefile
├── parser.y
├── README.md
└── test-programs
    ├── 01-declaracoes-simples.larp
    ├── 02-identificadores.larp
    ├── 03-prefixos-de-reservadas.larp
    ├── 04-espacos-e-quebras.larp
    ├── 05-inicializacao.larp
    ├── 06-if-else.larp
    ├── 07-ternario.larp
    ├── 08-route-fall.larp
    ├── 09-literais.larp
    ├── 11-caracteres-invalidos.larp
    ├── 12-numeros-malformados.larp
    ├── 13-strings-invalidas.larp
    ├── 21-tipo-inexistente.larp
    ├── 22-declaracao-sem-virgula.larp
    ├── 23-lista-incompleta.larp
    ├── 24-identificador-com-digito.larp
    ├── 25-if-sem-parenteses.larp
    ├── 26-inicializacao-sem-valor.larp
    ├── 27-route-sem-to.larp
    ├── 28-fim-inesperado.larp
    ├── 31-reatribuicao-e-fall-fora-de-case.larp
    ├── 32-comparacao-invalida.larp
   

## 2. Instalar o Flex e o Bison e compilar

O Colab ja vem com `gcc`, `g++` e `make`; apenas o `flex` e o `bison` precisam ser
instalados. Sao gerados dois binarios: `bin/parser` (lexer + parser) e `bin/lexer`
(so o lexer). Se a gramatica tiver conflitos, o Bison avisa aqui na compilacao.

In [11]:
!apt-get -qq install -y flex bison
!make
!ls -l bin/

flex -o bin/lex.yy.c clarker.l
bison -d -o bin/parser.tab.c parser.y
g++ -DSOMENTE_LEXER bin/lex.yy.c -o bin/lexer
g++ bin/parser.tab.c bin/lex.yy.c -o bin/parser
total 392
-rwxr-xr-x 1 root root  58008 Nov 12 02:47 lexer
-rw-r--r-- 1 root root  60428 Nov 12 02:47 lex.yy.c
-rwxr-xr-x 1 root root 180328 Nov 12 02:47 parser
-rw-r--r-- 1 root root  84934 Nov 12 02:47 parser.tab.c
-rw-r--r-- 1 root root   5035 Nov 12 02:47 parser.tab.h


## 3. Rodar o parser em todos os programas de teste

Passa o parser em cada arquivo de `test-programs/`. A dezena do numero do arquivo indica
o que o teste pretende exercitar:

| Faixa | Categoria        | Saida esperada do parser                                |
|-------|------------------|---------------------------------------------------------|
| `0x`  | validos          | traducao pos-fixada e mensagem de sucesso               |
| `1x`  | erros lexicos    | erros lexicos ate o primeiro erro sintatico (cascatas)  |
| `2x`  | erros sintaticos | um erro sintatico, com a linha                          |
| `3x`  | erros semanticos | todos os erros semanticos, com a linha                  |

In [12]:
!make test

=== test-programs/01-declaracoes-simples.larp ===

TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    LINHA  ESCOPO   USOS
   1  a                          int         1       0      0
   2  x                          float       2       0      0
   3  y                          float       2       0      0
   4  z                          float       2       0      0
   5  nome                       string      3       0      0
   6  _contador                  int         4       0      0
   7  total2                     int         4       0      0

TRADUCAO (notacao pos-fixada)
  (nenhum comando com expressao)

[LARP] Programa sintatica e semanticamente correto

=== test-programs/02-identificadores.larp ===
[LARP] ERRO SEMANTICO na linha 6: variavel 'a' ja declarada neste escopo (linha 5)

TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    LINHA  ESCOPO   USOS
   1  _                          int         1       0      0
   2  __x                        float      

## 4. Rodar so o lexer em todos os programas de teste

Roda os mesmo arquivos mas só com o lexer. Os testes `2x` e `3x` passam nesse caso com 0 erros, já que
sao lexicamente validos.

In [13]:
!make test-lexer

=== test-programs/01-declaracoes-simples.larp ===
TIPO(int) ID(a) PONTO_VIRGULA 
TIPO(float) ID(x) VIRGULA ID(y) VIRGULA ID(z) PONTO_VIRGULA 
TIPO(string) ID(nome) PONTO_VIRGULA 
TIPO(int) ID(_contador) VIRGULA ID(total2) PONTO_VIRGULA 

TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    1a OCORRENCIA  OCORRENCIAS
   1  a                          int                 1            1
   2  x                          float               2            1
   3  y                          float               2            1
   4  z                          float               2            1
   5  nome                       string              3            1
   6  _contador                  int                 4            1
   7  total2                     int                 4            1

RESUMO: 18 tokens, 7 simbolos, 0 erros


=== test-programs/02-identificadores.larp ===
TIPO(int) ID(_) PONTO_VIRGULA 
TIPO(float) ID(__x) VIRGULA ID(x_2) VIRGULA ID(total2) PONTO_VIRGULA 
TIPO(strin

## 5. Rodar um programa específico

Mostra o programa, a saida do parser e a sequencia de tokens do lexer. Basta trocar o
`ARQUIVO`.

In [14]:
ARQUIVO = 'test-programs/06-if-else.larp'

!cat $ARQUIVO
print('-' * 60)
!make run FILE=$ARQUIVO
print('-' * 60)
!make run-lexer FILE=$ARQUIVO

int a, b = 1, 2;
if (a < b) {
    a = a + 1;
} else {
    b = b - 1;
}
if (a<=b) { if (a>b) { a=b; } }
if (a >= b * 2) { int c = (a + b) / 2; }
if (a == b) { } else { a = a * (b - 1); }
if (a != 0) { b = b / a; }
------------------------------------------------------------

TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    LINHA  ESCOPO   USOS
   1  a                          int         1       0     13
   2  b                          int         1       0     12
   3  c                          int         8       1      0

TRADUCAO (notacao pos-fixada)
  linha   1  declaracao  a 1 =
  linha   1  declaracao  b 2 =
  linha   2  if          a b <
  linha   3  atribuicao  a a 1 + =
  linha   5  atribuicao  b b 1 - =
  linha   7  if          a b <=
  linha   7  if          a b >
  linha   7  atribuicao  a b =
  linha   8  if          a b 2 * >=
  linha   8  declaracao  c a b + 2 / =
  linha   9  if          a b ==
  linha   9  atribuicao  a a b 1 - * =
  linha  10  if         

O parser aponta a linha e o que era esperado em um programa com erro sintatico:

In [15]:
ARQUIVO = 'test-programs/25-if-sem-parenteses.larp'

!cat $ARQUIVO
print('-' * 60)
!make run FILE=$ARQUIVO

int a, b = 1, 2;
if a < b {
    a = b;
}
------------------------------------------------------------
[LARP] ERRO SINTATICO na linha 2: syntax error, unexpected ID, expecting (


Em um programa sintaticamente correto, mas com erros semanticos, o parser reporta todos os
erros semanticos encontrados, cada um com a linha:

In [16]:
ARQUIVO = 'test-programs/34-route-erro.larp'

!cat $ARQUIVO
print('-' * 60)
!make run FILE=$ARQUIVO

float f = 1.5;
int n = 2;
string s = "a";

route f to {
    1: {
        n = 1;
    }
}

route n to {
    1: {
        n = 10;
    }
    "dois": {
        n = 20;
    }
    s: {
        n = 30;
    }
}
------------------------------------------------------------
[LARP] ERRO SEMANTICO na linha 5: route nao aceita expressao do tipo float (use int ou string)
[LARP] ERRO SEMANTICO na linha 15: valor de case do tipo string, mas o route e sobre int
[LARP] ERRO SEMANTICO na linha 18: valor de case do tipo string, mas o route e sobre int

TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    LINHA  ESCOPO   USOS
   1  f                          float       1       0      1
   2  n                          int         2       0      5
   3  s                          string      3       0      1

[LARP] Estrutura sintaticamente correta, mas com 0 erro(s) lexico(s) e 3 erro(s) semantico(s)


## 6. Criar um novo programa de teste

Exemplo para criar um novo programa de teste para testar aqui imediatamente.

In [17]:
%%writefile test-programs/exemplo.larp
int limite, faixa;
float media;

limite = 10 + 5 * 2;
faixa = limite / 10;

if (limite > 20) {
    media = 1.5;
} else {
    media = 0.5;
}

route faixa to {
    1, 2: { limite = limite + 1; fall; }
    3: { limite = 0; }
} else {
    limite = 1;
}

Overwriting test-programs/exemplo.larp


In [18]:
!make run FILE=test-programs/exemplo.larp
print('-' * 60)
!make run-lexer FILE=test-programs/exemplo.larp


TABELA DE SIMBOLOS
   #  IDENTIFICADOR              TIPO    LINHA  ESCOPO   USOS
   1  limite                     int         1       0      7
   2  faixa                      int         1       0      2
   3  media                      float       2       0      2

TRADUCAO (notacao pos-fixada)
  linha   4  atribuicao  limite 10 5 2 * + =
  linha   5  atribuicao  faixa limite 10 / =
  linha   7  if          limite 20 >
  linha   8  atribuicao  media 1.5 =
  linha  10  atribuicao  media 0.5 =
  linha  13  route       faixa
  linha  14  atribuicao  limite limite 1 + =
  linha  15  atribuicao  limite 0 =
  linha  17  atribuicao  limite 1 =

[LARP] Programa sintatica e semanticamente correto
------------------------------------------------------------
TIPO(int) ID(limite) VIRGULA ID(faixa) PONTO_VIRGULA 
TIPO(float) ID(media) PONTO_VIRGULA 

ID(limite) ATRIBUICAO LITERAL_INT(10) MAIS LITERAL_INT(5) VEZES LITERAL_INT(2) PONTO_VIRGULA 
ID(faixa) ATRIBUICAO ID(limite) DIVIDIDO LITERAL_INT(